In [1]:
from boututils.datafile import DataFile
from boutdata.collect import collect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, pathlib
import platform
import traceback
import xarray as xr
import xbout
from pathlib import Path
import xhermes as xh

sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/sdtool_load_test/sdtools"))
sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/transients"))
sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/general_functions"))


from plotting_functions import *
from convergence_functions import * 

from hermes3.case_db import *
from hermes3.casedeck import*
from hermes3.load import *
from hermes3.named_selections import *
from hermes3.plotting import *
from hermes3.grid_fields import *
from hermes3.accessors import *
from hermes3.utils import *
from hermes3.fluxes import *
from hermes3.selectors import *
from hermes3.front_tracking import *
# from hermes3.balance1d import *

# plt.style.use('ggplot')
plt.rcParams.update({'font.size': 10})
linewidth = 3
markersize = 15



# plt.style.use('ggplot')
plt.style.use('default')
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 1
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['axes.grid'] = True
plt.rcParams.update({'font.size': 16})



%load_ext autoreload
%autoreload 2


In [20]:
parent_dir = '/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-06_slow_down_tests/case_restart'

to_load = [f'{parent_dir}/cold_start_new_build_eos', f'{parent_dir}/cold_start_old_diags_build']
cs = dict()

names= ['neutral-eos(2db1a8d)', 'next(9a42f6a)']

for i, j in enumerate(to_load):

    print(i)
    name = names[i]
    data = Path(j)
    # name = f"{i}"
    print(f"Loading {name}")
    cs[name] = Load.case_1D(data, guard_replace = False, use_squash=True)

0
Loading neutral-eos(2db1a8d)
- Looking for squash file
- Squash file found. squash date 06/19/2025, 14:50:28, dmp file date 06/19/2025, 12:05:20
Skipping unnormalisation
1
Loading next(9a42f6a)
- Looking for squash file
- Squash file found. squash date 06/19/2025, 14:52:26, dmp file date 06/19/2025, 15:45:47
- dmp files are newer than the squash file! Recreating...
- Done
Skipping unnormalisation


In [30]:
import numpy as np
import matplotlib.pyplot as plt
import imageio
import os


def detachment_front_index(ds, time=None):
    """
    Function to find the location of the detachment in the simulation
    """
    if time == None:
        ds = ds.isel(t=-1)

    else:
        ds = ds.isel(t=time)

    # Get the detachment location
    try:
        detachment_idx = np.where(ds['Te'][2:-2] < 5)[0][0]
    except:
        # If the detachment index is not found, return None
        print("No detachment found")
        detachment_idx = 0

    return detachment_idx

def detatchment_var(ds, var, time_range = (0, -1), save_gif = True, save_mp4 = True, static_scale = False, S_range = None, animation_name = "animation", output_dir = None, mp4_name = "animation.mp4"):
    import warnings
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    frames = []
    if output_dir is None:
        output_dir = "/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/transients/1_D/2025_updates/gif_frames"
    else:
        output_dir = output_dir
    os.makedirs(output_dir, exist_ok=True)

    # Loop over time indices
    for i in np.linspace(time_range[0], len(ds['t'][time_range[0]:time_range[1]]) - 1, 100, dtype=int):
        print(f"Creating frame {i}")
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ds_temp = ds.isel(t=i)
        
        det_idx = detachment_front_index(ds, time=i)
        y_flipped = np.max(ds['y']) - ds['y']
        
        ax.plot(y_flipped['y'][::-1], ds_temp[var], marker = 'o', markersize = 5)
        # ax.set_xbound(0, ds['y'][::-1][det_idx] + 3)
        ax.set_title(f"Time index: {i} \n name = {animation_name}")



        ax.set_xlabel('Distance from target (cm)')
        ax.set_ylabel(f'{var} ({ds[var].units})')

        if static_scale:
            ax.set_ybound(np.min(ds['NVd']) * 1.1,np.max(ds['NVd']) * 1.1)

        if S_range == None:
            # print(ds['y'].values[::-1][det_idx] + 3)
            
            ax.set_xbound(y_flipped[det_idx] + -3, ds['y'][::-1][det_idx] + 3)

        else:
            ax.set_xbound(0 , S_range)
        # ax.set_yscale('symlog')

        frame_path = f"{output_dir}/frame_{i:03d}.png"
        plt.savefig(frame_path)
        plt.close(fig)

        frames.append(imageio.imread(frame_path))

    # Save the gif
    if save_gif:
        print(f"Saving gif to {output_dir}/{animation_name}.gif")
        imageio.mimsave(f"{animation_name}.gif", frames, duration=0.4)

    # Read the gif using imageio
    gif_path = f'{animation_name}.gif'
    mp4_path = f'{mp4_name}.mp4'

    # Use imageio to convert gif to mp4
    with imageio.get_writer(mp4_path, format='mp4', fps=2) as writer:
        print(f"Converting {gif_path} to {mp4_path}")
        gif = imageio.mimread(gif_path)  # Read gif frames
        for frame in gif:
            # Convert to RGB (3 channels)
            frame_rgb = np.array(frame)
            
            # If the frame has 4 channels (RGBA), convert it to RGB (ignore the alpha channel)
            if frame_rgb.shape[-1] == 4:
                frame_rgb = frame_rgb[..., :3]
            
            # If it's grayscale (2D array), convert to 3 channels (RGB)
            elif frame_rgb.ndim == 2:
                frame_rgb = np.stack([frame_rgb] * 3, axis=-1)
            
            writer.append_data(frame_rgb)

            


In [31]:
print(cs['neutral-eos(2db1a8d)'].ds)

# for i in cs.values():
detatchment_var(cs['neutral-eos(2db1a8d)'].ds, 'NVd', time_range = (0, -1), save_gif = True, save_mp4 = True, static_scale = False, S_range = None, animation_name = "neutral-eos(2db1a8d)", output_dir = None, mp4_name = "neutral-eos(2db1a8d)")

<xarray.Dataset>
Dimensions:                        (pos: 804, t: 121)
Coordinates:
    dx                             (pos) float64 dask.array<chunksize=(804,), meta=np.ndarray>
    dy                             (pos) float64 dask.array<chunksize=(804,), meta=np.ndarray>
    dz                             (pos) float64 dask.array<chunksize=(804,), meta=np.ndarray>
  * t                              (t) float64 0.0 0.0009999 ... 0.119 0.12
    y                              (pos) float64 0.09755 0.2925 ... 82.4 82.41
  * pos                            (pos) float64 -0.2924 -0.09737 ... 82.02
Data variables: (12/89)
    Bxy                            (pos) float64 dask.array<chunksize=(804,), meta=np.ndarray>
    Ed+_iz                         (t, pos) float64 dask.array<chunksize=(121, 804), meta=np.ndarray>
    Ed+_rec                        (t, pos) float64 dask.array<chunksize=(121, 804), meta=np.ndarray>
    Ed_target_recycle              (t, pos) float64 dask.array<chunksize=(121

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[swscaler @ 0x7438880] Warning: data is not aligned! This can lead to a speed loss


In [ ]:
detatchment_var(cs[''].ds, 'NVd', time_range = (0, -1), save_gif = True, save_mp4 = True, static_scale = False, S_range = None, animation_name = "neutral_oscillation", output_dir = None, mp4_name = "neutral_oscillation.mp4")
